# 3rd EDA · STEP 1 — 전처리 및 정규화 (보고서 3종 발행 기업)**입력** `모델 점수 결과 3종 보고서 기업.xlsx` → 시트 `입력 모델 점수 결과`**대상** 사업보고서 · 기업지배구조보고서 · 지속가능경영보고서를 **모두 발행한** 기업-연도**출력** `ESG_3종발행_분석데이터_정규화.csv`---### 왜 3종 발행 기업만 보는가앞선 EDA에서 확인한 가장 큰 교란은 **공시량 편향**이었다.같은 KCGS 등급 안에서도 지속가능경영보고서를 낸 기업이 전체 +17.8점, E +18.2점을 더 받았고,회귀에서도 등급(R²=0.425)보다 보고서 발행 여부(R²=0.510)가 점수를 더 잘 설명했다.원인은 단순하다. RAG가 읽을 문서가 없으면 근거를 찾지 못해 최하단계로 채점된다.이것은 기업의 ESG 성과가 아니라 **입력 데이터의 유무**다.3종을 모두 발행한 기업으로 한정하면 **입력 조건이 동일해진다.**그래야 남은 점수 차이를 모델의 판단력으로 해석할 수 있다.### 이 노트북의 단계| 단계 | 내용 ||---|---|| 1 | 로드 및 원시 스키마 || 2 | 정제 6단계 || 3 | **3종 발행 필터 — 라벨과 플래그 정합성 검증** || 4 | 품질 검사 (결측 / 중복 / 범위 / 논리 일관성) || 5 | 분포 진단 + **전체 표본과 비교 (범위 제한 확인)** || 6 | **이상치 정밀 검토 — 제거할 것인가** || 7 | 정규화 필요성 진단 (영역 · 연도 · 시장 · 업종) || 8 | 정규화 5종 생성 || 9 | 방법별 효과 비교 || 10 | 권장안 확정 및 저장 |> **미리 알아둘 것** — Min-Max·Z-score·Robust는 모두 선형변환이라 상관계수를 바꾸지 않는다.> 정규화의 목적은 상관을 높이는 것이 아니라 **영역 간 비교 가능성**을 확보하는 것이다. 9절에서 수치로 확인한다.

## 0. 환경 설정

In [ ]:
# 한글 폰트 (코랩 최초 1회, 약 30초). 런타임 재시작 불필요.
!apt-get -qq install -y fonts-nanum > /dev/null
!fc-cache -f > /dev/null

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm._load_fontmanager(try_read_cache=False)
matplotlib.rc("font", family="NanumGothic")
matplotlib.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
print("폰트:", matplotlib.rcParams["font.family"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. 데이터 로드- **A. 직접 업로드** (기본값) — 실행 후 `모델 점수 결과 3종 보고서 기업.xlsx` 선택- **B. 구글 드라이브** — `USE_DRIVE = True` 로 바꾸고 `DRIVE_PATH` 수정

In [ ]:
USE_DRIVE  = False
DRIVE_PATH = "/content/drive/MyDrive/3rdEDA/모델 점수 결과 3종 보고서 기업.xlsx"
SHEET      = "입력 모델 점수 결과"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    XLSX = DRIVE_PATH
else:
    from google.colab import files
    XLSX = list(files.upload().keys())[0]

raw = pd.read_excel(XLSX, sheet_name=SHEET, dtype={"기업코드": str})
print("원본 shape:", raw.shape)
raw.head(3)

In [ ]:
schema = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "결측": raw.isna().sum(),
    "결측률(%)": (raw.isna().mean() * 100).round(1),
    "고유값": raw.nunique(),
    "예시": [raw[c].dropna().iloc[0] if raw[c].notna().any() else None for c in raw.columns],
})
display(schema)

## 2. 정제| # | 문제 | 처리 ||---|---|---|| 2-1 | 기업코드가 `103590.0` 형태 | 소수부 제거 후 6자리 zero-fill || 2-2 | 같은 코드에 사명 2개 | 최신 사명으로 통일 || 2-3 | 점수 컬럼에 문자열(`상폐` 등) | 숫자 변환 || 2-4 | 등급이 문자 | KCGS 7단계 순서값(1~7) 매핑 || 2-5 | 범주형 앞뒤 공백 (`둘 다 발행 `) | strip + 표기 통일 || 2-6 | 발행 플래그가 float/결측 | 0/1 정수화 |

In [ ]:
# ── 상수 ─────────────────────────────────────────────────
ORD  = {"D": 1, "C": 2, "B": 3, "B+": 4, "A": 5, "A+": 6, "S": 7}   # KCGS 7단계
LAB  = {v: k for k, v in ORD.items()}

GCOLS = ["ESG등급", "환경", "사회", "지배구조"]
SCOLS = ["전체결과", "P결과", "E결과", "S결과", "G결과"]
ECOLS = ["E결과", "S결과", "G결과"]
PAIRS = [("전체", "ESG등급", "전체결과"), ("환경(E)", "환경", "E결과"),
         ("사회(S)", "사회", "S결과"), ("지배구조(G)", "지배구조", "G결과")]

df = raw.copy()
print(f"시작 {len(df)}행")

In [ ]:
# ── 2-1. 기업코드 ────────────────────────────────────────
before = df["기업코드"].head(3).tolist()
df["기업코드"] = (df["기업코드"].astype(str)
                          .str.replace(r"\.0$", "", regex=True)
                          .str.strip().str.zfill(6))
print("2-1", before, "→", df["기업코드"].head(3).tolist(),
      "| 자릿수", df["기업코드"].str.len().value_counts().to_dict())

# ── 2-2. 사명 통일 ───────────────────────────────────────
nun = df.groupby("기업코드")["기업명"].nunique()
for c in nun[nun > 1].index:
    print("2-2 사명 중복", c, sorted(df.loc[df["기업코드"] == c, "기업명"].unique()))
df["기업명"] = df["기업명"].replace({"디티알오토모티브": "DN오토모티브"})   # 사명 변경, 동일 법인
print("    → 잔여 중복", int((df.groupby("기업코드")["기업명"].nunique() > 1).sum()), "건")

In [ ]:
# ── 2-3. 점수 숫자화 ─────────────────────────────────────
nonnum = df[df["전체결과"].notna() & pd.to_numeric(df["전체결과"], errors="coerce").isna()]
print(f"2-3 숫자가 아닌 점수 {len(nonnum)}행")
if len(nonnum):
    display(nonnum[["기업명", "기업코드", "평가년도", "전체결과", "비고"]])
for c in SCOLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ── 2-4. 등급 순서화 ─────────────────────────────────────
print("\n2-4 등급 원본 값")
for g in GCOLS:
    vals = sorted(df[g].dropna().unique().tolist())
    unknown = set(vals) - set(ORD)
    print(f"    {g:6s} {vals}" + (f"   !! 매핑불가 {unknown}" if unknown else ""))
    df[g + "_ord"] = df[g].map(ORD)

In [ ]:
# ── 2-5. 범주형 정리 ─────────────────────────────────────
print("2-5 보고서_발행_유형 (정리 전)")
print(df["보고서_발행_유형"].dropna().map(repr).value_counts().to_string())
df["보고서_발행_유형"] = (df["보고서_발행_유형"].str.strip()
                                    .replace({"지속가능경영": "지속가능경영만"}))
df["시장구분"] = df["시장구분"].str.strip()
print("\n(정리 후)")
print(df["보고서_발행_유형"].dropna().value_counts().to_string())

# ── 2-6. 발행 플래그 정수화 ──────────────────────────────
df["SR"] = pd.to_numeric(df["지속가능경영보고서_발행"], errors="coerce").fillna(0).astype(int)
df["CG"] = pd.to_numeric(df["기업지배구조보고서_발행"], errors="coerce").fillna(0).astype(int)
df["평가년도"] = pd.to_numeric(df["평가년도"], errors="coerce").astype("Int64")
print("\n2-6 SR/CG 정수화 완료:", df["SR"].value_counts().to_dict(), df["CG"].value_counts().to_dict())

## 3. 3종 발행 필터**3종** = 사업보고서 + 기업지배구조보고서 + 지속가능경영보고서사업보고서는 상장사 의무공시라 별도 플래그가 없다. 따라서 판정 기준은 `SR == 1 and CG == 1` 이다.원본에는 `보고서_발행_유형` 라벨과 `SR`/`CG` 플래그가 함께 있으므로,**둘이 일치하는지 먼저 확인한 뒤** 플래그를 기준으로 필터링한다.

In [ ]:
# 라벨 vs 플래그 정합성 (여기서 어긋나면 원본을 먼저 고쳐야 한다)
print("■ 보고서_발행_유형 × (SR, CG) 교차표")
ct = pd.crosstab(df["보고서_발행_유형"].fillna("(결측)"), [df["SR"], df["CG"]])
display(ct)

mismatch = df[(df["보고서_발행_유형"] == "둘 다 발행") & ~((df.SR == 1) & (df.CG == 1))]
print(f"라벨은 '둘 다 발행'인데 플래그가 아닌 행: {len(mismatch)}건")
mismatch2 = df[((df.SR == 1) & (df.CG == 1)) & (df["보고서_발행_유형"] != "둘 다 발행")]
print(f"플래그는 둘 다 1인데 라벨이 다른 행: {len(mismatch2)}건")

In [ ]:
# 단계별 필터링
m_score = df["전체결과"].notna()
m_grade = df["ESG등급_ord"].notna()
m_three = (df.SR == 1) & (df.CG == 1)

steps = [("전체", len(df)),
         ("채점 점수 있음", int(m_score.sum())),
         ("KCGS 등급 있음", int(m_grade.sum())),
         ("채점 + 등급", int((m_score & m_grade).sum())),
         ("3종 발행", int(m_three.sum())),
         ("채점 + 등급 + 3종 (최종)", int((m_score & m_grade & m_three).sum()))]
display(pd.DataFrame(steps, columns=["단계", "행 수"]).set_index("단계"))

d = df[m_score & m_grade & m_three].copy().reset_index(drop=True)
d["보고서수"] = d["SR"] + d["CG"]

print(f"\n최종 분석 대상: {len(d)}행 / {d['기업코드'].nunique()}개 기업 / "
      f"{sorted(d['평가년도'].dropna().unique().tolist())}")
print("시장구분:", d["시장구분"].value_counts().to_dict())

In [ ]:
print("■ 연도별 커버리지")
display(d.groupby("평가년도").agg(기업수=("기업코드", "nunique"), 행수=("전체결과", "size")).T)

print("■ 기업별 보유 연도 수")
display(d.groupby("기업코드").size().value_counts().sort_index()
         .rename_axis("보유 연도").rename("기업 수").to_frame().T)

print("■ 대상 기업 목록")
lst = (d.groupby(["기업코드", "기업명"])
         .agg(연도수=("평가년도", "count"),
              연도=("평가년도", lambda s: ",".join(str(int(x)) for x in sorted(s))))
         .reset_index())
display(lst)

## 4. 품질 검사

In [ ]:
print("■ 결측")
display(d[SCOLS + [g + "_ord" for g in GCOLS]].isna().sum().to_frame("결측").T)

print("■ 범위 이탈 (0~100)")
print({c: int(((d[c] < 0) | (d[c] > 100)).sum()) for c in SCOLS})

print("\n■ 중복 (기업코드 × 평가년도):", int(d.duplicated(["기업코드", "평가년도"]).sum()), "건")

print("\n■ 논리 일관성 — 전체결과가 P/E/S/G 가중평균과 맞는가")
# 91항목 = 정보공시 5 + 환경 32 + 사회 34 + 지배구조 20 (항목당 동일가중)
W = {"P결과": 5, "E결과": 32, "S결과": 34, "G결과": 20}
recon = sum(d[c] * w for c, w in W.items()) / sum(W.values())
gap = (d["전체결과"] - recon).abs()
print(f"    최대 오차 {gap.max():.4f}점 / 평균 {gap.mean():.5f}점 / 0.5점 초과 {(gap > .5).sum()}건")
print("    → 전체결과는 영역 점수의 항목수 가중평균이 맞다 (독립 산식 아님)")

## 5. 분포 진단 — 전체 표본과 비교

In [ ]:
rows = []
for c in SCOLS:
    x = d[c].dropna()
    q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1
    sh = stats.shapiro(x)
    rows.append({"컬럼": c, "n": len(x), "평균": x.mean(), "표준편차": x.std(),
                 "CV": x.std() / x.mean(), "최소": x.min(), "중앙값": x.median(), "최대": x.max(),
                 "왜도": stats.skew(x), "첨도": stats.kurtosis(x),
                 "Shapiro p": sh.pvalue, "정규성": "기각" if sh.pvalue < .05 else "유지",
                 "|z|>3": int((np.abs((x - x.mean()) / x.std()) > 3).sum()),
                 "IQR 이상치": int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())})
dist = pd.DataFrame(rows).set_index("컬럼")
display(dist.round(4))

In [ ]:
# 전체 표본(3종 필터 이전)과 비교 — 무엇이 달라졌는가
full = df[m_score & m_grade]
cmp = pd.DataFrame({
    "전체표본 평균": full[SCOLS].mean(), "3종 평균": d[SCOLS].mean(),
    "전체표본 SD":  full[SCOLS].std(),  "3종 SD":  d[SCOLS].std(),
})
cmp["평균 변화"] = (cmp["3종 평균"] - cmp["전체표본 평균"]).round(2)
cmp["SD 비율"]  = (cmp["3종 SD"] / cmp["전체표본 SD"]).round(3)
display(cmp.round(2))

print("등급 분포 비교 (ESG등급)")
gcmp = pd.DataFrame({"전체표본": full["ESG등급"].value_counts(),
                     "3종":     d["ESG등급"].value_counts()}).reindex(list(ORD)[::-1]).fillna(0).astype(int)
display(gcmp.T)

> **범위 제한(range restriction)이 발생했다.**> 3종 발행 기업만 남기면 점수 평균이 올라가고(E 30.8→39.1, 전체 40.7→48.4)> 표준편차는 줄어든다(전체 SD 0.75배, S 0.74배). 등급 분포도 상위로 쏠린다(D 18→2건).>> 표본의 산포가 좁아지면 상관계수는 **구조적으로 낮게** 나온다.> 따라서 이후 STEP 2에서 상관이 떨어져 보여도, 그것이 (a) 공시 교란이 걷힌 진짜 값인지> (b) 범위 제한 때문인지 구분해야 한다. 아래 셀에서 보정치를 함께 계산해 둔다.

In [ ]:
# 범위 제한 보정 (Thorndike Case II) — 참고용
print("■ 범위 제한 보정 참고치")
rows = []
for name, g, s in PAIRS:
    Sg, sg = full[g + "_ord"].std(ddof=1), d[g + "_ord"].std(ddof=1)
    r = stats.pearsonr(d[g + "_ord"], d[s]).statistic
    k = Sg / sg
    rows.append({"영역": name, "등급 SD(전체→3종)": f"{Sg:.2f}→{sg:.2f}",
                 "축소비 k": round(k, 3), "관측 r": round(r, 3),
                 "보정 r": round(r * k / np.sqrt(1 + r ** 2 * (k ** 2 - 1)), 3)})
display(pd.DataFrame(rows).set_index("영역"))
print("※ 선택이 제3변수(공시 조건)에 걸려 있어 근사치다. 절대값이 아니라 '얼마나 과소평가되는가'의 감으로만 쓴다.")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(19, 6.5))
for j, c in enumerate(SCOLS):
    x, xf = d[c].dropna(), full[c].dropna()
    axes[0, j].hist(xf, bins=18, alpha=.45, label="전체표본", color="#BBBBBB")
    axes[0, j].hist(x,  bins=18, alpha=.75, label="3종 발행", color="#4C78A8")
    axes[0, j].axvline(x.mean(), color="crimson", lw=1.4)
    axes[0, j].set_title(c); axes[0, j].legend(fontsize=7); axes[0, j].grid(alpha=.25)
    stats.probplot(x, dist="norm", plot=axes[1, j])
    axes[1, j].set_title(f"{c} Q-Q"); axes[1, j].grid(alpha=.25)
    axes[1, j].get_lines()[0].set_markersize(3)
axes[0, 0].set_ylabel("빈도"); axes[1, 0].set_ylabel("표본 분위수")
plt.tight_layout(); plt.show()

## 6. 이상치 정밀 검토전체 표본에서는 이상치가 0건이었지만, 3종 하위표본에서는 산포가 좁아지면서 IQR 기준 이상치가 나타난다.**기계적으로 지우지 말고 실제 행을 확인한다.** 채점 오류라면 제외, 실제로 낮은 기업이라면 유지한다.

In [ ]:
for c in SCOLS:
    x = d[c]
    q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    o = d[(x < lo) | (x > hi)]
    if len(o) == 0:
        print(f"■ {c}: 이상치 없음")
        continue
    print(f"■ {c}: {len(o)}건  (하한 {lo:.1f} / 상한 {hi:.1f})")
    display(o[["기업명", "평가년도", "ESG등급", c, "보고서_발행_유형"]]
            .sort_values(c).reset_index(drop=True))

> **판정: 이상치를 제거하지 않는다.**>> - `P결과` 12건은 정보공시 5개 항목만의 평균이라 값이 25점 단위로 뚝뚝 끊긴다.>   상한이 100점에 붙어 있고 분포가 강하게 좌왜(왜도 −1.83)라 IQR 규칙이 하한을 과하게 잡아낸 것이다.> - `전체결과`·`S결과`의 소수 건은 모두 **C·B 등급 기업**이다. 등급과 방향이 일치하므로 오류가 아니라 실제 저점이다.> - `|z| > 3` 기준으로는 P결과 1건뿐이고, E·G는 두 기준 모두 0건이다.>> 즉 제거 대상이 되는 '측정 오류'는 없다. 대신 P결과가 비정규·이산적이라는 점을 기록해 두고,> 상관 분석에서는 순위 기반(Spearman)을 기본으로 쓴다.

## 7. 정규화 필요성 진단관행적으로 스케일링하지 않고, 이 표본에 **어떤 이질성이 실제로 있는지** 먼저 확인한다.

In [ ]:
# 진단 1 — 영역 간 스케일 이질성
print("■ 진단 1  E/S/G 척도가 서로 다른가")
display(d[ECOLS].agg(["mean", "std", "min", "max"]).T.round(2))
lev = stats.levene(d["E결과"], d["S결과"], d["G결과"])
fri = stats.friedmanchisquare(d["E결과"], d["S결과"], d["G결과"])
print(f"  Levene(분산)   p = {lev.pvalue:.3g}  → {'분산 다름' if lev.pvalue < .05 else '차이 없음'}")
print(f"  Friedman(평균) p = {fri.pvalue:.3g}  → {'평균 다름' if fri.pvalue < .05 else '차이 없음'}")
print("\n  판정: 표준화 필요 ✅")

In [ ]:
# 진단 2 — 연도 효과
print("■ 진단 2  연도에 따라 점수 수준이 다른가")
display(d.groupby("평가년도")[SCOLS].mean().round(2))
for c in SCOLS:
    H, p = stats.kruskal(*[v[c].values for _, v in d.groupby("평가년도")])
    print(f"  {c:6s} Kruskal H={H:5.2f} p={p:.4f}  {'연도 효과 있음' if p < .05 else '연도 효과 없음'}")
print("\n  판정: 연도내 표준화 불필요 ❌  (3종 발행 기업만 보면 연도 간 평균이 48점대로 거의 평평하다)")

In [ ]:
# 진단 3 — 시장구분 / 업종
print("■ 진단 3-a  시장구분")
print(d["시장구분"].value_counts().to_dict(), "→ 단일 시장이면 통제 불필요")

print("\n■ 진단 3-b  업종 (산업코드 앞 4자리)")
d["업종4"] = d["산업코드"].astype(str).str[:4]
vc = d["업종4"].value_counts()
display(vc.rename("행 수").to_frame().T)
big = vc[vc >= 5].index
sub = d[d["업종4"].isin(big)]
print(f"n>=5 인 업종 {len(big)}개 / {len(sub)}행 으로 검정")
for c in SCOLS:
    H, p = stats.kruskal(*[v[c].values for _, v in sub.groupby("업종4")])
    print(f"  {c:6s} Kruskal H={H:5.2f} p={p:.4f}  {'업종 효과 있음' if p < .05 else '-'}")
print("\n  판정: 업종 효과는 다섯 점수 모두에서 유의하다 ⚠️  그럼에도 업종내 표준화는 채택하지 않는다.")
print("        (a) 19개 산업코드 중 n=1~4인 업종이 절반이라 그룹 평균·표준편차가 불안정하고,")
print("        (b) 업종 차이의 일부는 실제 ESG 성과 차이일 수 있어 지우면 신호까지 사라진다.")
print("        → STEP 2에서 '표준화 대상'이 아니라 '통제변수'로 다룬다 (업종 고정효과 / 층화 비교).")

## 8. 정규화 / 표준화 생성

In [ ]:
# how: z | minmax | robust | pct.  by가 주어지면 그 그룹 안에서 계산한다.
def add_scaled(data, cols, by=None, suffix="", how="z"):
    out = data.copy()
    if how == "z":
        f = lambda v: (v - v.mean()) / v.std(ddof=1)
    elif how == "minmax":
        f = lambda v: (v - v.min()) / (v.max() - v.min()) * 100
    elif how == "robust":
        f = lambda v: (v - v.median()) / (v.quantile(.75) - v.quantile(.25))
    elif how == "pct":
        f = lambda v: v.rank(pct=True) * 100
    else:
        raise ValueError(how)
    for c in cols:
        out[c + suffix] = out.groupby(by)[c].transform(f) if by else f(out[c])
    return out


d = add_scaled(d, SCOLS, how="minmax", suffix="_mm")    # ① Min-Max 0~100
d = add_scaled(d, SCOLS, how="z",      suffix="_z")     # ② Z-score (전체)
d = add_scaled(d, SCOLS, how="robust", suffix="_rb")    # ③ Robust (중앙값/IQR)
d = add_scaled(d, SCOLS, how="pct",    suffix="_pct")   # ④ 백분위 (전체)
d = add_scaled(d, SCOLS, how="z",      suffix="_zy", by="평가년도")   # ⑤ 연도내 Z (참고용)

# 등급도 같은 척도로 (잔차 비교용)
for g in GCOLS:
    d[g + "_ord_z"] = (d[g + "_ord"] - d[g + "_ord"].mean()) / d[g + "_ord"].std(ddof=1)

print("생성 컬럼", len([c for c in d.columns if c.endswith(("_mm", "_z", "_rb", "_pct", "_zy"))]), "개")
display(d[["기업명", "평가년도", "E결과", "E결과_mm", "E결과_z", "E결과_rb", "E결과_pct", "E결과_zy"]]
        .head(6).round(3))

In [ ]:
# 변환 결과가 의도한 성질을 갖는지 확인
chk = pd.DataFrame({
    "원점수 (평균, SD)":   [tuple(np.round([d[c].mean(), d[c].std()], 2)) for c in ECOLS],
    "MinMax (최소, 최대)": [tuple(np.round([d[c + "_mm"].min(), d[c + "_mm"].max()], 1)) for c in ECOLS],
    "Z (평균, SD)":       [tuple(np.round([d[c + "_z"].mean(), d[c + "_z"].std()], 3)) for c in ECOLS],
    "Robust (중앙, IQR)": [tuple(np.round([d[c + "_rb"].median(),
                                          d[c + "_rb"].quantile(.75) - d[c + "_rb"].quantile(.25)], 3)) for c in ECOLS],
    "백분위 (최소, 최대)":   [tuple(np.round([d[c + "_pct"].min(), d[c + "_pct"].max()], 1)) for c in ECOLS],
}, index=["E", "S", "G"])
display(chk)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
sets = [("원점수", ECOLS), ("Min-Max 0~100", [c + "_mm" for c in ECOLS]),
        ("Z-score", [c + "_z" for c in ECOLS]), ("백분위", [c + "_pct" for c in ECOLS])]
for ax, (title, cols) in zip(axes, sets):
    ax.boxplot([d[c].dropna() for c in cols], showmeans=True)
    ax.set_xticklabels(["E", "S", "G"])
    ax.set_title(title); ax.grid(axis="y", alpha=.3)
plt.suptitle("영역 간 척도 정렬 — 원점수는 E(39.1)와 G(56.9)가 어긋나 있다", y=1.05)
plt.tight_layout(); plt.show()

## 9. 방법별 효과 비교

In [ ]:
methods = [("원점수", ""), ("① Min-Max", "_mm"), ("② Z-score(전체)", "_z"),
           ("③ Robust", "_rb"), ("④ 백분위(전체)", "_pct"), ("⑤ Z-score(연도내)", "_zy")]
rr, rs = [], []
for label, suf in methods:
    a, b = {"변환": label}, {"변환": label}
    for name, g, s in PAIRS:
        x = d[[g + "_ord", s + suf]].dropna()
        a[name] = round(stats.pearsonr(x[g + "_ord"], x[s + suf]).statistic, 4)
        b[name] = round(stats.spearmanr(x[g + "_ord"], x[s + suf]).statistic, 4)
    rr.append(a); rs.append(b)

print("■ Pearson r")
display(pd.DataFrame(rr).set_index("변환"))
print("■ Spearman ρ")
display(pd.DataFrame(rs).set_index("변환"))

> **확인** — ① Min-Max · ② Z-score · ③ Robust 는 원점수와 소수 4자리까지 동일하다.> 선형변환이므로 당연한 결과이며, **정규화로 상관이 좋아지지 않는다**는 사실을 여기서 못 박아 둔다.> ④ 백분위는 Spearman이 같고 Pearson만 소폭 변한다(순위 보존).> ⑤ 연도내 Z는 값이 미세하게 흔들리지만 진단 2에서 연도 효과가 없었으므로 채택하지 않는다.

In [ ]:
# 표준화가 실제로 열어주는 것 ① — 영역 간 상대 강약
prof = d.groupby(["기업코드", "기업명"])[[c + "_z" for c in ECOLS]].mean().round(2)
prof.columns = ["E(z)", "S(z)", "G(z)"]
prof["E-G 격차"] = (prof["E(z)"] - prof["G(z)"]).round(2)
print("■ 같은 기업 안에서 어느 영역이 상대적으로 약한가 (원점수로는 볼 수 없다)")
print("\n[E가 G보다 약함]"); display(prof.nsmallest(5, "E-G 격차"))
print("[G가 E보다 약함]"); display(prof.nlargest(5, "E-G 격차"))

In [ ]:
# 표준화가 실제로 열어주는 것 ② — 등급과 점수를 같은 축에 놓기
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, g, s) in zip(axes, PAIRS):
    gz, sz = d[g + "_ord_z"], d[s + "_z"]
    ax.scatter(gz, sz, s=22, alpha=.55, edgecolor="none")
    ax.plot([-3, 3], [-3, 3], "--", color="gray", lw=1, label="완전 일치선")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_title(f"{name}   RMSE={np.sqrt(((gz - sz) ** 2).mean()):.3f}")
    ax.set_xlabel("KCGS 등급 (z)"); ax.grid(alpha=.3)
axes[0].set_ylabel("채점 점수 (z)"); axes[0].legend(fontsize=8)
plt.suptitle("표준화하면 등급과 점수의 어긋난 정도를 같은 자로 잴 수 있다", y=1.03)
plt.tight_layout(); plt.show()

for name, g, s in PAIRS:
    diff = d[s + "_z"] - d[g + "_ord_z"]
    print(f"  {name:10s} RMSE={np.sqrt((diff ** 2).mean()):.3f}  "
          f"|차이|>1.5σ: {(diff.abs() > 1.5).sum()}건 ({(diff.abs() > 1.5).mean():.1%})")

## 10. 권장안 및 저장### 채택| 용도 | 컬럼 | 이유 ||---|---|---|| **기본 분석** | `*_z` | 영역 간 비교 가능, 평균 0·SD 1로 해석이 명확 || **보고·시각화** | `*_pct` | "상위 몇 %"로 바로 읽힘, 비정규 분포에 안전 || 원본 확인 | 원점수 | 채점 결과 자체를 검증할 때 || 참고 보관 | `*_mm`, `*_rb`, `*_zy` | 팀이 다른 방식을 요구할 때 대비 (기본 분석에는 미사용) |### 하지 않기로 한 것| 처리 | 판단 | 근거 ||---|---|---|| 이상치 제거 | ❌ | 채점 오류 아님. P결과는 25점 단위 이산값, 나머지는 등급과 방향 일치 || 로그·제곱근 변환 | ❌ | 꼬리가 두꺼운 게 아니라 좌왜. 변환해도 순위 불변이라 실익 없음 || 연도내 표준화 | ❌ | 연도 효과 없음 (Kruskal p=0.48~0.96) || 업종내 표준화 | ❌ | 업종 효과는 **유의하지만**(p=0.0005~0.037) n=1~4 업종이 절반이라 그룹 통계가 불안정. STEP 2에서 통제변수로 다룬다 || 시장구분 통제 | 불필요 | 대상 105행이 전부 KOSPI || 공시유형 그룹내 표준화 | 불필요 | 3종 발행으로 필터링해 이미 조건이 동일 |### 이 표본의 성격 (STEP 2에서 반드시 감안할 것)- **공시 조건이 통제되었다** — 이전 EDA 최대 교란이던 보고서 유무가 상수가 되었다.- **대신 범위 제한이 생겼다** — 등급 SD가 0.75~0.91배로 줄어 상관이 구조적으로 낮게 나온다.- **상위 편중** — ESG등급 A 이상이 57/105(54%), D는 2건뿐이다. 하위 구간 판별력은 이 표본으로 평가할 수 없다.

In [ ]:
OUTCSV = "ESG_3종발행_분석데이터_정규화.csv"

keep = (["기업명", "기업코드", "평가년도", "시장구분", "산업코드", "업종4",
         "보고서_발행_유형", "SR", "CG", "보고서수", "시가총액"]
        + GCOLS + [g + "_ord" for g in GCOLS] + [g + "_ord_z" for g in GCOLS]
        + SCOLS
        + [c + "_z"   for c in SCOLS]
        + [c + "_pct" for c in SCOLS]
        + [c + "_mm"  for c in SCOLS]
        + [c + "_rb"  for c in SCOLS]
        + [c + "_zy"  for c in SCOLS])
keep = [c for c in keep if c in d.columns]

final = d[keep].sort_values(["기업코드", "평가년도"]).reset_index(drop=True)
final.to_csv(OUTCSV, index=False, encoding="utf-8-sig")

print(f"저장: {OUTCSV}")
print(f"  {final.shape[0]}행 × {final.shape[1]}열 / {final['기업코드'].nunique()}개 기업")
print(f"  결측: {final.isna().sum().sum()}건 "
      f"(시가총액 {int(final['시가총액'].isna().sum())}건 — 2025년 행에만 값이 있다)")
display(final.head(3))

# 코랩에서 내려받기
# from google.colab import files; files.download(OUTCSV)

In [ ]:
print("=" * 64)
print("3rd EDA · STEP 1 완료 요약")
print("=" * 64)
print(f"입력             {len(raw)}행")
print(f"분석 대상        {len(final)}행 / {final['기업코드'].nunique()}개 기업 / 2022~2025 / 전부 KOSPI")
print(f"필터 기준        SR=1 AND CG=1 (사업 + 지배구조 + 지속가능경영 = 3종)")
print(f"제외             채점 없음 · 등급 없음 · 3종 미달")
print(f"이상치 제거      0건 (검토 후 유지 — 측정 오류 아님)")
print(f"분포 변환        없음")
print(f"채택 표준화      Z-score(전체) + 백분위(전체)")
print(f"기각 표준화      연도내 Z / 업종내 Z / 공시유형 그룹내 Z")
print()
print("STEP 2 예고 — 이 표본에서 확인할 것")
print("  · 공시 교란을 걷어낸 뒤에도 등급-점수 상관이 남는가")
print("  · 업종 효과 통제 (Kruskal p<0.05로 유의 — 고정효과 또는 층화)")
print("  · 판별타당도: E/S/G를 구분해서 재고 있는가")
print("  · 범위 제한 보정 후 실제 상관 수준")
print("  · 기업내 연도 변화 추적력 (동일 기업 4개년 15개사)")